# Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import LanusStats as ls
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
import hdbscan
from sklearn.metrics import silhouette_score
import pickle
import os

sofascore = ls.SofaScore()

In [ ]:
import json
import time
from bs4 import BeautifulSoup
from faker import Faker
import undetected_chromedriver as uc

CHROME_MAJOR = 146

def patched_sofascore_request(self, path):
    path = f"{self.base_url}{path}"

    chrome_options = uc.ChromeOptions()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--ignore-certificate-errors")
    chrome_options.add_argument("--ignore-ssl-errors")
    chrome_options.add_argument("--allow-insecure-localhost")
    chrome_options.add_argument("--disable-web-security")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--window-size=1920,1080")

    fake = Faker()
    chrome_options.add_argument(f"user-agent={fake.chrome()}")

    driver = uc.Chrome(options=chrome_options, version_main=CHROME_MAJOR)

    try:
        driver.get(path)
        time.sleep(3)
        html = driver.page_source
    finally:
        driver.quit()

    soup = BeautifulSoup(html, "html.parser")
    data = json.loads(soup.text)
    time.sleep(2)
    return data

ls.SofaScore.sofascore_request = patched_sofascore_request
sofascore = ls.SofaScore()

# Scrap de datos — todas las posiciones

In [ ]:
ligas = {
    'col': 'Colombia Primera A Apertura',
    'arg': 'Argentina Liga Profesional',
    'bra': 'Brasileirão Série A',
    'bol': 'Bolivia Division Profesional',
    'chi': 'Chile Primera Division',
    'ecu': 'Ecuador LigaPro',
    'per': 'Peru Liga 1',
    'uru': 'Uruguay Primera Division',
    'ven': 'Venezuela Primera Division',
    'mls': 'MLS'
}

# ✅ Loop por las 4 posiciones
positions = ['Goalkeepers', 'Defenders', 'Midfielders', 'Forwards']

dfs = []

for codigo, liga in ligas.items():
    for position in positions:
        print(f"Scraping: {liga} — {position}")
        try:
            df = sofascore.scrape_league_stats(
                liga,
                '2026',
                selected_positions=[position]
            )
            df['league_code'] = codigo
            df['league_name'] = liga
            df['season'] = '2026'
            df['position_group'] = position
            dfs.append(df)
        except Exception as e:
            print(f"  ⚠️ Error en {liga} — {position}: {e}")
            continue

df_2026 = pd.concat(dfs, ignore_index=True)
print(f"\n✅ Total jugadores: {len(df_2026)}")
df_2026['position_group'].value_counts()

# Definición de features por posición (Opción B)

In [ ]:
# ============================================================
# FEATURES POR POSICIÓN
# Cada posición tiene sus propias métricas relevantes
# ============================================================
FEATURES_POR_POSICION = {
    'Goalkeepers': [
        'saves_p90',
        'savePercentage',
        'cleanSheet',
        'goalsConcededPerGame',
        'accurateLongBalls_p90',
        'accurateLongBallsPercentage',
        'accuratePasses_p90',
        'accuratePassesPercentage',
    ],
    'Defenders': [
        'interceptions_p90',
        'groundDuelsWon_p90',
        'groundDuelsWonPercentage',
        'totalDuelsWon_p90',
        'totalDuelsWonPercentage',
        'accurateLongBalls_p90',
        'accurateLongBallsPercentage',
        'accuratePasses_p90',
        'accuratePassesPercentage',
        'dribbledPast_p90',
        'fouls_p90',
    ],
    'Midfielders': [
        'keyPasses_p90',
        'accurateFinalThirdPasses_p90',
        'accuratePasses_p90',
        'accuratePassesPercentage',
        'interceptions_p90',
        'groundDuelsWon_p90',
        'dribbles_p90',
    ],
    'Forwards': [
        'goals_p90',
        'assists_p90',
        'shots_p90',
        'keyPasses_p90',
        'dribbles_p90',
        'bigChancesCreated_p90',
        'accurateFinalThirdPasses_p90',
        'wasFouled_p90',
    ]
}

# Ingeniería de features

In [ ]:
df = df_2026.copy()

# Columnas que necesitan dividirse por minutos
per90_map = {
    'goals_p90':                    'goals',
    'assists_p90':                  'assists',
    'shots_p90':                    'totalShots',
    'keyPasses_p90':                'keyPasses',
    'interceptions_p90':            'interceptions',
    'duelsWon_p90':                 'totalDuelsWon',
    'totalDuelsWon_p90':            'totalDuelsWon',
    'dribbles_p90':                 'successfulDribbles',
    'bigChancesCreated_p90':        'bigChancesCreated',
    'accurateFinalThirdPasses_p90': 'accurateFinalThirdPasses',
    'accuratePasses_p90':           'accuratePasses',
    'groundDuelsWon_p90':           'groundDuelsWon',
    'wasFouled_p90':                'wasFouled',
    'passToAssist_p90':             'passToAssist',
    'accurateLongBalls_p90':        'accurateLongBalls',
    'fouls_p90':                    'fouls',
    'dribbledPast_p90':             'dribbledPast',
    'dispossessed_p90':             'dispossessed',
    'saves_p90':                    'saves',
}

for p90_col, raw_col in per90_map.items():
    if raw_col in df.columns:
        df[p90_col] = df[raw_col] / df['minutesPlayed'] * 90

# Columnas de porcentaje: forzar numérico
pct_cols = [
    'accuratePassesPercentage',
    'groundDuelsWonPercentage',
    'totalDuelsWonPercentage',
    'accurateCrossesPercentage',
    'accurateLongBallsPercentage',
    'savePercentage',
]
for col in pct_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Reemplazar infinitos y nulos
df = df.replace([np.inf, -np.inf], np.nan)

print("✅ Features calculadas")
df.shape

# KMeans por posición

In [ ]:
MIN_MINUTES = 600
N_CLUSTERS = 6

scalers = {}
kmeans_models = {}
df_model_list = []

for position, features in FEATURES_POR_POSICION.items():
    print(f"\n--- {position} ---")

    # Filtrar por posición y minutos mínimos
    df_pos = df[(df['position_group'] == position) & (df['minutesPlayed'] > MIN_MINUTES)].copy()

    # Solo features que existen en el DataFrame
    available_features = [f for f in features if f in df_pos.columns]
    print(f"  Jugadores: {len(df_pos)} | Features disponibles: {len(available_features)}/{len(features)}")

    df_pos = df_pos.replace([None, np.inf, -np.inf], np.nan)
    df_pos = df_pos.dropna(subset=available_features).copy()
    print(f"  Jugadores tras limpiar NaN: {len(df_pos)}")

    if len(df_pos) < N_CLUSTERS:
        print(f"  ⚠️ Muy pocos jugadores para clustering, se omite.")
        continue

    X = df_pos[available_features]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    df_pos['cluster_kmeans'] = kmeans.fit_predict(X_scaled)
    df_pos['features_used'] = str(available_features)

    scalers[position] = scaler
    kmeans_models[position] = kmeans
    df_model_list.append(df_pos)
    print(f"  ✅ Clustering completado")

df_model = pd.concat(df_model_list, ignore_index=True)
print(f"\n✅ Total jugadores en modelo: {len(df_model)}")

# Exportar datos y modelos

In [ ]:
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Guardar CSV principal
df_model.to_csv('data/jugadores_2026.csv', index=False)
print("✅ data/jugadores_2026.csv guardado")

# Guardar scalers y modelos KMeans por posición
with open('models/scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

with open('models/kmeans_models.pkl', 'wb') as f:
    pickle.dump(kmeans_models, f)

# Guardar diccionario de features por posición
with open('models/features_por_posicion.pkl', 'wb') as f:
    pickle.dump(FEATURES_POR_POSICION, f)

print("✅ models/scalers.pkl guardado")
print("✅ models/kmeans_models.pkl guardado")
print("✅ models/features_por_posicion.pkl guardado")
print("\n🚀 Todo listo para la app de Streamlit")